### 📂 Excel to Terms Text Extractor

This script extracts keyword terms from an **Excel file** and saves them into `.txt` files for further processing.

**Workflow:**
1. Reads all sheets from the given Excel file.
2. For each sheet:
   - Selects the **`term_column`** (or defaults to the first column).
   - Cleans the values (drops empty cells, converts to string).
   - Saves terms as a comma-separated list in `<sheetname>.txt`.
3. After all sheets are processed:
   - Creates a **`combined.txt`** file with all terms (deduplicated).
4. Outputs are stored in the specified `output_dir`.

**Parameters:**
- `excel_path`: Path to your Excel file.
- `output_dir`: Folder to save generated `.txt` files (default: current directory).
- `term_column`: Column name containing terms (default: first column).

In [7]:
# @title Imports and configurations

import os
import pandas as pd
from typing import Optional

In [8]:
# @title Core functionality

def extract_terms_to_txt(
    excel_path: str,
    output_dir: Optional[str] = None,
    term_column: Optional[str] = None
):
    """
    Reads each sheet from the given Excel file and writes its terms to
    a separate .txt file (terms comma-separated), and also creates
    combined.txt with all terms from all sheets.

    Parameters:
    - excel_path: Path to the Excel file.
    - output_dir: Directory where .txt files will be saved (default: current directory).
    - term_column: Name of the column containing terms (default: first column).
    """
    # Prepare output directory
    base_dir = output_dir or os.getcwd()
    os.makedirs(base_dir, exist_ok=True)

    xls = pd.ExcelFile(excel_path)
    all_terms = []

    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet)

        # Select the term column (or first column if not specified/found)
        if term_column and term_column in df.columns:
            series = df[term_column]
        else:
            series = df.iloc[:, 0]

        # Clean and collect terms
        terms = series.dropna().astype(str).tolist()
        all_terms.extend(terms)

        # Write per‑sheet file
        sheet_file = os.path.join(base_dir, f"{sheet}.txt")
        with open(sheet_file, 'w', encoding='utf-8') as f:
            f.write(','.join(terms))

    # After all sheets: write combined file
    combined_file = os.path.join(base_dir, "combined.txt")
    with open(combined_file, 'w', encoding='utf-8') as f:
        # Remove duplicates if desired: uncomment the next line
        all_terms = list(dict.fromkeys(all_terms))
        f.write(','.join(all_terms))

In [9]:
# @title Use example
extract_terms_to_txt(
    excel_path='/content/openfSMR_July_2025_groups_splitted.xlsx',
    output_dir='terms_txt',
    term_column='Term'  # or leave out if your terms are in the first column
)

In [13]:
# @title Save Results as a zip archive
import shutil

folder_path = "/content/terms_txt"
zip_path = "/content/terms_txt.zip"

# Create zip
shutil.make_archive("/content/terms_txt", 'zip', folder_path)

# Download
from google.colab import files
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 🧾BioPortal Ontology Recommender Pipeline

This script automates ontology recommendations using the **BioPortal Recommender API** for sets of domain-specific terms.

**Workflow:**
1. Reads `.txt` files from `TERMS_DIR` (each file contains comma-separated terms). Those files can be created from `.xlsx`, see **Excel to Terms Text Extractor**
2. Sends the terms (in batches) to the BioPortal Recommender API.
3. Collects ontology acronyms recommended for each file.
4. Compares each file’s results against the results from `combined.txt` (the file containign all terms).
   - Reports **overlapped** (common) and **non_overlapped** ontologies.
5. Saves all results in `recommendations.json`.

**Parameters:**
- Requires a valid `bioportal-key`. Currently is taken from the stored Colab keys
- Adjust constants (`BATCH_SIZE`, `MIN_VALID`, `EXCLUDE`) if needed.


- Output JSON is saved inside the same folder as the terms.

In [10]:
# @title Imports and configurations

import os
import json
import requests
from typing import List, Dict, Iterable, Optional
from google.colab import userdata

# ─── Configuration ─────────────────────────────────────────────────────────────
API_KEY = userdata.get('bioportal-key')
TERMS_DIR = "/content/terms_txt"     # directory containing *.txt files
OUTPUT_JSON = os.path.join(TERMS_DIR, "recommendations.json")
BATCH_SIZE = 200 # batch of the terms sent
MIN_VALID = 6 # minimum list of the reutrned ontologies
EXCLUDE = set() # excluded ontologies  e.g. {"GO", "CHEBI"}

In [11]:
# @title Core functionality

# ─── Helper Functions ──────────────────────────────────────────────────────────
def read_terms(filepath: str) -> List[str]:
    """Read comma‑separated terms from a text file."""
    with open(filepath, encoding="utf-8") as f:
        content = f.read()
    return [t.strip() for t in content.split(",") if t.strip()]

def chunk_terms(terms: List[str], chunk_size: int) -> Iterable[List[str]]:
    """Yield successive chunks of size `chunk_size`."""
    for i in range(0, len(terms), chunk_size):
        yield terms[i : i + chunk_size]

def recommend_ontology_acronyms(
    terms: Iterable[str],
    apikey: str,
    *,
    min_valid: int = MIN_VALID,
    exclude: Optional[set] = EXCLUDE
) -> List[str]:
    """
    Call BioPortal Recommender for a batch of terms.
    Returns up to `min_valid` ontology acronyms (sorted by batch score).
    """
    params = {
        "input": ",".join(terms),
        "input_type": 2,
        "apikey": apikey,
    }
    try:
        resp = requests.get(
            "https://data.bioontology.org/recommender",
            params=params,
            timeout=240
        )
        resp.raise_for_status()
        recs = resp.json()
    except requests.RequestException as e:
        print(f"[ERROR] Request failed for batch: {e}")
        return []

    results: List[str] = []
    for entry in recs:
        try:
            ont = entry["ontologies"][0]
            acr = ont["acronym"]
        except (KeyError, IndexError):
            continue
        if acr in exclude:
            continue
        if acr not in results:
            results.append(acr)
            if len(results) >= min_valid:
                break
    return results

def aggregate_for_file(filepath: str) -> List[str]:
    """Read terms, run in batches, and return the *aggregated* list of acronyms."""
    terms = read_terms(filepath)
    batch_outs = []
    for batch in chunk_terms(terms, BATCH_SIZE):
        batch_outs.extend(recommend_ontology_acronyms(batch, API_KEY))
    # Deduplicate, preserving order
    seen = set()
    aggregated = []
    for acr in batch_outs:
        if acr not in seen:
            seen.add(acr)
            aggregated.append(acr)
    return aggregated


In [12]:
# @title Use example

all_txt = sorted(f for f in os.listdir(TERMS_DIR) if f.endswith(".txt"))
if "combined.txt" not in all_txt:
    raise FileNotFoundError("combined.txt not found in " + TERMS_DIR)

# 2) Process combined.txt first
combined_path = os.path.join(TERMS_DIR, "combined.txt")
print("Processing combined.txt …")
combined_ontos = aggregate_for_file(combined_path)

# 3) Initialize result structure
results: Dict[str, object] = {
    "combined.txt": combined_ontos
}

# 4) Process each other file
for fname in all_txt:
    if fname == "combined.txt":
        continue
    path = os.path.join(TERMS_DIR, fname)
    print(f"Processing {fname} …")
    ontos = aggregate_for_file(path)

    overlapped     = [o for o in ontos if o in combined_ontos]
    non_overlapped = [o for o in ontos if o not in combined_ontos]

    results[fname] = {
        "overlapped":     overlapped,
        "non_overlapped": non_overlapped
    }

# 5) Write to JSON
with open(OUTPUT_JSON, "w", encoding="utf-8") as jf:
    json.dump(results, jf, indent=2)

print(f"\nAll done! Recommendations saved to:\n  {OUTPUT_JSON}")
print("Structure:")
print(json.dumps(results, indent=2)[:500] + "\n…")


Processing combined.txt …


KeyboardInterrupt: 